# Viettel AI Race - Train NER trên Colab

Notebook này dùng gói `viettel_colab_data.zip` được tạo từ repo local. Quy trình:

1. Upload gói data.
2. Kiểm tra GPU và dữ liệu thủ công.
3. Train thử trên split train/val để chọn epoch.
4. Train lại trên toàn bộ 100 file.
5. Chạy inference và tải file nộp.

Khuyến nghị runtime: **L4 hoặc A100 GPU**. Nếu bị OOM với `xlm-roberta-large`, đổi `MODEL_NAME` sang `xlm-roberta-base` và chạy lại.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || true

# Giữ torch có sẵn của Colab để tránh tải lại CUDA quá nặng; chỉ pin các thư viện NLP.
!pip -q install -U "transformers==4.44.2" "accelerate==1.10.1" "sentencepiece==0.2.2" "numpy==1.26.4"

import torch, transformers, numpy as np
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('bf16:', torch.cuda.is_bf16_supported())

## 1. Upload gói data

Ở máy local, chạy:

```bash
bash colab/pack.sh
```

Sau đó upload file `artifacts/viettel_colab_data.zip` ở ô dưới.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os, shutil

uploaded = files.upload()
zip_names = [name for name in uploaded if name.endswith('.zip')]
assert zip_names, 'Bạn cần upload file artifacts/viettel_colab_data.zip'
zip_name = zip_names[0]

work = Path('/content/vietel')
if work.exists():
    shutil.rmtree(work)
work.mkdir(parents=True)

with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(work)

os.chdir(work)
print('Đã giải nén vào', work)
!find . -maxdepth 2 -type d | sort
!wc -l data/ner/*.jsonl

## 2. Kiểm tra dữ liệu thủ công

Ô này kiểm tra nhanh: đủ 100 file input, đủ 100 file nhãn tay, offset đúng, và phân bố nhãn hợp lý.

In [ ]:
import json
from pathlib import Path
from collections import Counter

input_dir = Path('input')
gt_dir = Path('data/gt_block')
assert len(list(input_dir.glob('*.txt'))) == 100, 'Thiếu input/*.txt'
assert len(list(gt_dir.glob('*.json'))) == 100, 'Thiếu data/gt_block/*.json'

counts = Counter()
bad = []
for fp in sorted(gt_dir.glob('*.json'), key=lambda p: int(p.stem)):
    raw = (input_dir / f'{fp.stem}.txt').read_text(encoding='utf-8')
    ents = json.loads(fp.read_text(encoding='utf-8'))
    for i, e in enumerate(ents):
        s, t = e['position']
        if raw[s:t] != e['text']:
            bad.append((fp.name, i, e['position'], e['text'], raw[s:t]))
        counts[e['type']] += 1

print('Tổng entity:', sum(counts.values()))
print(counts)
print('Offset lỗi:', len(bad))
assert not bad, bad[:3]

split = json.loads(Path('data/blocks/split.json').read_text(encoding='utf-8'))
print('train files:', len(split['train']['files']), '| val files:', len(split['val']['files']))

## 3. Rebuild data train/val

Gói upload đã có sẵn `data/ner`, nhưng chạy lại bước này để chắc chắn data dẫn xuất khớp với nhãn tay mới nhất.

In [ ]:
!python src/ner_data.py

## 4. Train thử trên train/val

Chọn `MODEL_NAME`:

- `xlm-roberta-large`: nên dùng nếu Colab có L4/A100, chất lượng tốt hơn.
- `xlm-roberta-base`: nhanh và nhẹ hơn, dùng khi bị OOM.

Sau khi chạy xong, xem bảng recall ở ô kế tiếp để chọn `EPOCHS_FINAL`.

In [ ]:
MODEL_NAME = 'xlm-roberta-large'  # đổi sang 'xlm-roberta-base' nếu bị OOM
EPOCHS_DEV = 20
BS = 4
ACCUM = 4
LR = '2e-5'
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
PRECISION_FLAG = '--bf16' if USE_BF16 else '--fp16'
print(MODEL_NAME, PRECISION_FLAG)

In [ ]:
!python src/ner_train.py   --model $MODEL_NAME   --out models/ner_dev   --epochs $EPOCHS_DEV   --bs $BS   --accum $ACCUM   --lr $LR   $PRECISION_FLAG   2>&1 | tee /content/train_dev.log

## 5. Xem epoch tốt nhất

In [ ]:
import ast, re
log = Path('/content/train_dev.log').read_text(encoding='utf-8', errors='ignore')
rows = []
for m in re.findall(r"\{'eval_loss'.*?\}", log):
    try:
        rows.append(ast.literal_eval(m))
    except Exception:
        pass

print(f"{'epoch':>7} {'recall':>8} {'precision':>10} {'f1':>8} {'loss':>9}")
for r in rows:
    print(f"{r.get('epoch', 0):7.1f} {r.get('eval_recall', 0):8.4f} {r.get('eval_precision', 0):10.4f} {r.get('eval_f1', 0):8.4f} {r.get('eval_loss', 0):9.4f}")

if rows:
    best = max(rows, key=lambda r: r.get('eval_recall', 0))
    print('
Epoch nên dùng cho train final:', round(best.get('epoch', EPOCHS_DEV)))
else:
    print('Không parse được log; nếu training hoàn tất, dùng epoch có eval_recall cao nhất trong log phía trên.')

## 6. Chấm offline trên validation

Đây là điểm theo nhãn tay của mình, dùng để so sánh giữa các model/checkpoint. Nó không phải điểm BTC tuyệt đối.

In [ ]:
!python src/ner_infer.py --model models/ner_dev --input input --out /content/pred_dev
!python src/score.py --gt data/gt_block --pred /content/pred_dev --pairing overlap --split val

## 7. Train final trên cả 100 file

Đổi `EPOCHS_FINAL` theo epoch tốt nhất ở bước 5. Với `--all`, validation đã nằm trong train nên số eval chỉ để kiểm tra code còn chạy, không dùng để chọn model nữa.

In [ ]:
EPOCHS_FINAL = 20  # sửa theo epoch tốt nhất ở bước 5
!python src/ner_train.py   --model $MODEL_NAME   --all   --out models/ner   --epochs $EPOCHS_FINAL   --bs $BS   --accum $ACCUM   --lr $LR   $PRECISION_FLAG   2>&1 | tee /content/train_final.log

## 8. Inference và đóng gói submission

In [ ]:
!python src/ner_infer.py --model models/ner --input input --out submission_ner

# Kiểm schema + offset trước khi zip.
import json
from pathlib import Path
bad = []
for fp in sorted(Path('submission_ner').glob('*.json'), key=lambda p: int(p.stem)):
    raw = Path('input', f'{fp.stem}.txt').read_text(encoding='utf-8')
    ents = json.loads(fp.read_text(encoding='utf-8'))
    for i, e in enumerate(ents):
        s, t = e['position']
        if raw[s:t] != e['text']:
            bad.append((fp.name, i, e['position']))
assert not bad, bad[:5]
print('Offset OK, files:', len(list(Path('submission_ner').glob('*.json'))))

# Dạng chính thức theo đề: output/1.json ... output/100.json
!rm -rf /content/output
!mkdir -p /content/output
!cp submission_ner/*.json /content/output/
!cd /content && zip -q -r output.zip output

# Dạng dự phòng: 100 json ở gốc zip, giống bản nộp thử từng được hệ thống nhận.
!cd submission_ner && zip -q -r /content/submission_root.zip .

!ls -lh /content/output.zip /content/submission_root.zip

## 9. Tải kết quả về máy

In [ ]:
from google.colab import files
files.download('/content/output.zip')
files.download('/content/submission_root.zip')

## 10. Tải weights về để nộp source code nếu vào top

File này có thể rất nặng. Chỉ tải khi bạn cần đóng gói source/weights cho BTC.

In [ ]:
!cd models && zip -q -r /content/ner_weights.zip ner
!ls -lh /content/ner_weights.zip
files.download('/content/ner_weights.zip')